# TP1 — Le réseau du métro de Montréal
## Carnet, graphes, Dijkstra — et un premier modèle

**Nom :** André GEORGES &nbsp;&nbsp;&nbsp; **Coéquipier :** _s'il y a lieu_ &nbsp;&nbsp;&nbsp; **Groupe :** 07979

---

Ce TP se fait **seul ou en équipe de 2**. Une équipe remet **un seul carnet** et
indique à la fin qui a fait quoi.

**Chaque question, y compris les questions de code, se termine par une cellule
d'observation en Markdown.** Écrivez-y ce que vous constatez, en phrases complètes,
en français. Une cellule d'observation laissée telle quelle vaut 0 pour la question.

Avant de remettre : **Kernel → Redémarrer et tout exécuter**.

## Partie 2 — Charger et explorer le réseau

Le fichier `donnees/metro_montreal.csv` contient une ligne par lien entre deux
stations voisines, avec la durée approximative en minutes. Le fichier
`donnees/stations.csv` donne la ligne de chaque station.

In [1]:
import csv

reseau = {}
liens = []

with open("donnees/metro_montreal.csv", encoding="utf-8") as fichier:
    for ligne in csv.DictReader(fichier):
        a = ligne["station_a"]
        b = ligne["station_b"]
        ligne1 = ligne["ligne"]
        minutes = int(ligne["minutes"])
        reseau.setdefault(a, []).append((b, minutes))
        reseau.setdefault(b, []).append((a, minutes))
        liens.append((a, b, ligne1))

infos_station = {}
with open("donnees/stations.csv", encoding="utf-8") as fichier:
    for ligne in csv.DictReader(fichier):
        infos_station[ligne["station"]] = ligne


print(len(reseau), "stations")


def voisins(reseau, station):
    return [nom for nom, minutes in reseau[station]]


def temps(reseau, a, b):
    for nom, minutes in reseau[a]:
        if nom == b:
            return minutes
    return None

68 stations


**Q2.1** — Affichez le nombre de stations et le nombre de **liens**.

In [2]:
# Q2.1
def nbLiens(reseau):
    nbLiens = []
    for station in reseau:
        nbLiens.append(len(reseau[station]))
    print(int(sum(nbLiens)/2), "liens")
        
print(len(reseau), "stations")
nbLiens(reseau)


68 stations
69 liens


> **Observation — obligatoire.** Comparez vos deux nombres. Qu'est-ce que leur écart vous dit sur la forme du réseau ?
>
> Dans un arbre, on s'attend à avoir un nombre de liens égal au nombre de noeuds moins un. 
> Ici, il y a plus de liens que de noeuds.
> Cela indique que le réseau n'est pas un arbre et qu'il contient au moins un cycle.

**Q2.2** — Trouvez la ou les stations qui ont le plus de voisins, puis la liste des
**terminus**.

In [3]:
# Q2.2
def nbVoisins(reseau):
    nbVoisins= 0
    stationMax= ""
    for station in reseau:
        if len(reseau[station]) > nbVoisins:
            nbVoisins = len(reseau[station])
            stationMax  = station
    print(stationMax, "a", nbVoisins, "voisins")
        
nbVoisins(reseau)

def estTerminus(liens):
    terminus = []
    for station in reseau:
        compteurs = {}
        for a, b, ligne in liens:
            if a == station or b  == station:
                compteurs[ligne] = compteurs.get(ligne, 0 ) + 1
        for ligne, n in compteurs.items():
            if n == 1:
                terminus.append(station)
    print("liste des terminus", terminus)
        
estTerminus(liens)


Berri-UQAM a 5 voisins
liste des terminus ['Angrignon', 'Berri-UQAM', 'Honore-Beaugrand', 'Cote-Vertu', 'Snowdon', 'Montmorency', 'Saint-Michel', 'Longueuil']


> **Observation — obligatoire.** Pourquoi ces stations-là ? Expliquez ce que leur nombre de voisins traduit dans le vrai réseau.
>
> il est important de définir ce qu'est un terminus:
> un terminus est défini par le fait qu'il n'y a qu'un voisin.
>
> d'ou la difficulté de le déterminer, puisque le terminus d'une ligne aboutit souvent a une correspondance d'une autre ligne, par exemple la ligne bleue à Snowdon, ou la ligne Jaune à Berri-Uqam, ce qui augmente le nombre de ses voisins.
>
> il faut donc d'abord déterminer le nombre de voisins sur sa propre ligne.

**Q2.3** — Comptez combien de tronçons durent 1, 2 et 3 minutes.

In [4]:
# Q2.3
def nbTroncons():
    uneMin = 0
    deuxMin = 0
    troisMin = 0
    total = 0
    for station in reseau:
        for voisin, minutes in reseau[station]:
            if minutes == 1:
                uneMin = uneMin + 1
            
            if minutes == 2:
                deuxMin = deuxMin + 1
            
            if minutes == 3:
                troisMin = troisMin + 1 
            total += 1
            
    print(
        "troncons de 1 minute:", int(uneMin / 2),
        "\ntroncons de 2 minute:", int(deuxMin / 2),
        "\ntroncons de 3 minute:", int(troisMin / 2),
        "\ntotal: ", int(total / 2)
    )
            
            
            
nbTroncons()
            
            

troncons de 1 minute: 13 
troncons de 2 minute: 37 
troncons de 3 minute: 19 
total:  69


> **Observation — obligatoire.** Quelle durée domine, et dans quelle proportion ? Gardez ce chiffre : il servira à la partie 5.
>
> La durée qui domine est:  2 minutes. le nombre de troncons de cette durée est à peu près deux fois plus grande que pour les autres tronçons.

## Partie 3 — Le parcours en largeur, et son arbre

**Q3.1** — Complétez `bfs(reseau, depart)`. Il rend `distance` (en nombre d'arrêts)
et `precedent`.

In [5]:
from collections import deque

def bfs(reseau, depart):
    distance = {depart: 0}
    precedent = {}
    file = deque([depart])
    while file:
        station = file.popleft()
        for suivante in voisins(reseau, station):
            if suivante not in distance:
                distance[suivante] = distance[station] + 1
                file.append(suivante)
                precedent[suivante]  = station

    return distance, precedent

distance, precedent = bfs(reseau, "Joliette")

for station, nb in distance.items():
    if nb == max(distance.values()):
        print( "station la plus loin:",  station) 
print(distance)

station la plus loin: Cote-Vertu
{'Joliette': 0, 'Prefontaine': 1, 'Pie-IX': 1, 'Frontenac': 2, 'Viau': 2, 'Papineau': 3, 'Assomption': 3, 'Beaudry': 4, 'Cadillac': 4, 'Berri-UQAM': 5, 'Langelier': 5, 'Saint-Laurent': 6, 'Champ-de-Mars': 6, 'Sherbrooke': 6, 'Jean-Drapeau': 6, 'Radisson': 6, 'Place-des-Arts': 7, "Place-d'Armes": 7, 'Mont-Royal': 7, 'Longueuil': 7, 'Honore-Beaugrand': 7, 'McGill': 8, 'Square-Victoria': 8, 'Laurier': 8, 'Peel': 9, 'Bonaventure': 9, 'Rosemont': 9, 'Guy-Concordia': 10, "Lucien-L'Allier": 10, 'Beaubien': 10, 'Atwater': 11, 'Georges-Vanier': 11, 'Jean-Talon': 11, 'Lionel-Groulx': 12, 'Jarry': 12, 'De Castelnau': 12, 'Fabre': 12, 'Charlevoix': 13, 'Place-Saint-Henri': 13, 'Cremazie': 13, 'Parc': 13, "D'Iberville": 13, 'LaSalle': 14, 'Vendome': 14, 'Sauve': 14, 'Acadie': 14, 'Saint-Michel': 14, "De l'Eglise": 15, 'Villa-Maria': 15, 'Henri-Bourassa': 15, 'Outremont': 15, 'Verdun': 16, 'Snowdon': 16, 'Cartier': 16, 'Edouard-Montpetit': 16, 'Jolicoeur': 17, 'Cote-

> **Observation — obligatoire.** Décrivez en vos mots ce que contient `precedent` à la fin, et à quoi il servira.
>
> "precedent" contient une liste de clé-valeurs; 
> la clé: la station actuelle
> la valeur: la station précédente
> ca permet de pouvoir passer de voisin en voisin et ainsi de parcourir l'arbre.
> je suppose que ca servira à trouver un chemin, puisqu'on peut aussi trouver les minutes des tronçons. 

**Q3.2** — Lancez le BFS depuis la station de votre choix. Combien de stations sont
atteintes ? Laquelle est la plus loin, en nombre d'arrêts ?

In [6]:
# Q3.2
depart =  "Joliette"
distance, precedent = bfs(reseau, depart)

station_loin = None

for station, nb in distance.items():
    if nb == max(distance.values()):
        station_loin = station

print( "station la plus loin:",  station)         
print("distance jusqu'a cette station:", max(distance.values()), "arrets")

station la plus loin: Cote-Vertu
distance jusqu'a cette station: 22 arrets


> **Observation — obligatoire.** Le nombre de stations atteintes vous apprend quelque chose sur le réseau : quoi ? Et la station la plus loin vous surprend-elle ?
>
>Le plus grand nombre que j'ai pu atteindre est en prenant une extrémité du réseau: 'Honore-Beaugrand' a 29 arrêts jusqu'à 'Cote-Vertu'

Le réseau a en tout 68 stations. Le nombre de 29 fait du sens, puisqu'il s,agit d'un réseau, avec des moyens plus courts d'aller d'un endroit à un autre. Je ne suis donc pas surpris. 

une seule ligne, par exemple d'un train intercity aurait forcément donné le maximum, soit 67 arrets.

**Q3.3** — Dessinez l'**arbre de parcours** (les 3 premiers niveaux) avec
`graphviz`.

In [7]:
from graphviz import Digraph
# Q3.3

dot = Digraph()

distance, precedent = bfs(reseau, "Joliette")

for station, precedent_station in precedent.items():
    if distance[station] < 10:
        dot.edge(precedent_station, station)
        
dot.render("arbre de parcours", directory="sorties", cleanup=True)



'sorties\\arbre de parcours.pdf'

> **Observation — obligatoire.** Le réseau n'est pas un arbre, et pourtant votre dessin en est un. Expliquez pourquoi.
>
> Le fait que ca soit un arbre est en effet discutable; 
On a au départ une (en fait plusieurs) ligne.s  

Mais le BFS construit une structure de relation parents-enfants à partir de la station de départ.

On aura donc dans la plupart des cas, des 'arbres' avec un seul enfant par parent. pour comparaison, j'ai aussi fait un graphviz d'une stations de correspondance, ici, Berri-UQAM.

quand une station a déjà été découverte par un autre chemin, elle n'est pas répétée; les cycles sont éliminés. 

par exemple Berri-UQAM et Lionel-Groulx (v. arbr de parcours_10niveaux.pdf) il y a une connection entre Georges-Vanier et Lionel-Groulx qui n'apparait pas ici. 

## Partie 4 — Dijkstra, en minutes

**Q4.1** — Complétez `plus_petite_case`, `relacher` et `dijkstra`.

In [8]:
INFINI = 9999

def plus_petite_case(distance, barres):
    record = INFINI
    choisie = None
    for station in distance:
        if station not in barres and distance[station] < record:
            record = distance[station]
            choisie = station
    return choisie


def relacher(distance, precedent, a, b, minutes):
    nouvelle = distance[a] + minutes
    if nouvelle < distance[b]:
        distance[b] = nouvelle
        precedent[b] = a


def dijkstra(reseau, depart):
    distance = {station: INFINI for station in reseau}
    distance[depart] = 0
    precedent = {}
    barres = set()
    while len(barres) < len(reseau):
        station = plus_petite_case(distance, barres)
        barres.add(station)
        for b in voisins(reseau, station):
            relacher(distance, precedent, station, b, temps(reseau, station, b))
            # print("etape", len(barres), ":", sommets[i], distance )
    return distance, precedent

distance, precedent = dijkstra(reseau, "Joliette")
print(distance["Cote-Vertu"])




41


> **Observation — obligatoire.** À quoi sert l'ensemble `barres` ? Que se passerait-il si on l'oubliait ?
>
>'barres' c'est un set() pour accumuler les stations que Dijkstra a déjà traitées.
>
> si on l'oubliait, et Si on n'avait pas 'if station not in barres',   plus_petite_case(distance, barres) sélectionnerait à nouveau une station déjà traitée.
> On n'en finirait pas de traiter les stations et on tomberait dans une boucle infinie.  

**Q4.2** — Complétez `chemin` (la remontée des précédents) et `duree`.

In [9]:
def chemin(precedent, depart, arrivee):
    route = [arrivee]
    while arrivee != depart:
        route.append(precedent[arrivee])
        arrivee = precedent[arrivee]
    route.reverse()
    return route


def duree(reseau, route):
    total = 0
    for i in range(len(route) - 1):
        total = total + temps(reseau, route[i], route[i+1])
    return total

arrivee= "Cremazie"
print(duree(reseau, chemin(precedent, depart, arrivee)))



25


> **Observation — obligatoire.** `chemin` remonte à l'envers puis retourne la liste. Pourquoi ne peut-on pas la construire à l'endroit directement ?
>
>  parce qu'on se sert de ce qu'on a: on a un dictionnaire de précédent qui nous donne la station précédente.
> si on partait de la station de départ, il n'y aurait pas moyen d'avoir la station suivante

**Q4.3** — Affichez le trajet le plus rapide entre deux stations de votre choix,
avec sa durée.

In [10]:
# Q4.3
depart = "Joliette"
arrivee = "Cremazie"

print(chemin(precedent, depart, arrivee))
print(duree(reseau, chemin(precedent, depart, arrivee)))

['Joliette', 'Prefontaine', 'Frontenac', 'Papineau', 'Beaudry', 'Berri-UQAM', 'Sherbrooke', 'Mont-Royal', 'Laurier', 'Rosemont', 'Beaubien', 'Jean-Talon', 'Jarry', 'Cremazie']
25


> **Observation — obligatoire.** Ce trajet correspond-il à celui que vous prendriez dans la vraie vie ? Si non, qu'est-ce qui manque à notre modèle ?
>
>il ne correspond pas.
> En comparant avec ce que me donne Google Map, je fais ces observations de ce qui manque à notre modèle:
> -Google Map calcule les temps de marche: 6 minutes de chez moi au métro. 1 minute pour Marcher à la ligne orange à Berri-UQAM, et le temps de marche de la station Crémazie au Collège Ahuntsic.
> -il manque aussi les temps d'attente du métro à chaque étape: temps d'attente à Joliette et à Berri-UQAM
> -sans compter les imprévus. Éventuelles interruptions de service. 
> Donc, un jour où il y a un examen, je ne m'y fierais pas et partirais à l'avance!

## Partie 5 — Comparer les deux, et surligner le chemin

**Q5.1** — Écrivez `comparer(depart, arrivee)` qui affiche, pour chaque algorithme,
le nombre d'arrêts **et** la durée. Testez-la sur **trois** paires de stations, dont
**au moins une** où les deux algorithmes ne donnent pas le même trajet.

In [11]:
# Q5.1
def comparer(depart, arrivee):
    distance_bfs, precedent_bfs = bfs(reseau, depart)
    distance_dijkstra, precedent_dijkstra = dijkstra(reseau, depart)
    
    route_bfs = chemin(precedent_bfs, depart, arrivee)
    route_dijkstra = chemin(precedent_dijkstra, depart, arrivee)
    
    nb_arrets_bfs = len(route_bfs) - 1
    nb_arrets_dijkstra = len(route_dijkstra) - 1
    
    duree_bfs = duree(reseau, route_bfs)
    duree_dijkstra = duree(reseau, route_dijkstra)
    
    print(f"{'BFS':<25}{'DIJKSTRA':<25}")
    print(f"{'Nb d\'arrets: ' + str(nb_arrets_bfs):<25}{'Nb d\'arrets: ' + str(nb_arrets_dijkstra):<25}")
    print(f"{'Duree: ' + str(duree_bfs) + " min":<25}{'Duree: ' + str(duree_dijkstra) + " min":<25}")
    print("=" *50)

comparer("Joliette", "Cremazie")
comparer("Parc", "Angrignon")
comparer("Namur", "Jean-Drapeau")





BFS                      DIJKSTRA                 
Nb d'arrets: 13          Nb d'arrets: 13          
Duree: 25 min            Duree: 25 min            
BFS                      DIJKSTRA                 
Nb d'arrets: 17          Nb d'arrets: 22          
Duree: 41 min            Duree: 37 min            
BFS                      DIJKSTRA                 
Nb d'arrets: 15          Nb d'arrets: 15          
Duree: 27 min            Duree: 27 min            


> **Observation — obligatoire.** Sur vos trois paires, à quelle fréquence les deux sont-ils d'accord ? Reliez votre réponse au décompte des durées de la question 2.3.
>
> Sur mes trois paires, les algorithmes sont d'accord dans deux cas sur trois:
> 
> J'ai eu la chance de tomber sur une situation intéressante:
> quand on utilise "Parc" et "Angrignon" comme points de départ et arrivée, on constate que:
> 
> BFS a choisi le chemin avec le moins d'arrêts(17), mais d'une durée plus longue(41min)
> 
> tandis que:
> DIJKSTRA a choisi le chemin avec le PLUS d'arrêts(22), mais d'une durée plus COURTE(37min) 

**Q5.2** — Sur votre paire divergente, **surlignez le chemin de Dijkstra** dans le
dessin : les arêtes du chemin en **rouge et en épais**, la durée de chaque tronçon en
étiquette, et les stations du chemin remplies d'une couleur.

> **Les attributs à utiliser.** Sur une arête : `color="red"`, `penwidth="3"`,
> `fontcolor="red"`. Sur un sommet : `style="filled"`, `fillcolor="#ffd9d9"`.
> Le reste — quelles arêtes dessiner, comment reconnaître celles du chemin — est à
> vous.

In [12]:
# Q5.2
depart = "Parc"
arrivee = "Angrignon"

distance, precedent = dijkstra(reseau, depart)

route = chemin(precedent,depart, arrivee)

print(route)

from graphviz import Digraph

dot = Digraph()

for station in route:
    dot.node(station, style="filled", fillcolor="#ffd9d9")

for i in range(len(route) - 1):
    a = route[i]
    b = route[i + 1]
    minutes = temps(reseau, a, b)
    
    dot.edge(
        a,
        b,
        color="red",
        penwidth="3",
        fontcolor="red",
        label=str(minutes)
    )
    
    dot.render("Q5.2", directory="sorties", cleanup=True)





['Parc', 'De Castelnau', 'Jean-Talon', 'Beaubien', 'Rosemont', 'Laurier', 'Mont-Royal', 'Sherbrooke', 'Berri-UQAM', 'Saint-Laurent', 'Place-des-Arts', 'McGill', 'Peel', 'Guy-Concordia', 'Atwater', 'Lionel-Groulx', 'Charlevoix', 'LaSalle', "De l'Eglise", 'Verdun', 'Jolicoeur', 'Monk', 'Angrignon']


> **Observation — obligatoire.** Décrivez ce que votre dessin rend visible et que la liste de stations ne montrait pas.
>
> On peut voir la durée entre chaque station.
> les étiquettes permettent de voir où se trouvent les tronçons de 1, 2 ou 3 minutes.
> Le dessin avec son trait rouge permet de voir le chemin de Dijkstra d'un seul coup d'oeil.
> Les flèches donnent la direction.

---
## Partie 6 — Construire un tableau de données

Un modèle a besoin d'un **tableau** : une ligne par exemple, une colonne par
**donnée**, et une colonne à part pour l'**étiquette**.

Ici, **un exemple = une station**, et l'étiquette est sa **ligne**. Les données,
vous les fabriquez avec **votre propre travail des parties 3 et 4**.

On laisse de côté les stations de la ligne jaune : il y en a trop peu.

**Q6.1** — Pour chaque station retenue, fabriquez la liste de ses mesures :

1. son nombre de voisins ;
2. la durée moyenne de ses tronçons ;
3. la durée de son plus long tronçon ;
4. le nombre d'**arrêts** qui la séparent d'un **repère** *(votre BFS)* ;
5. le nombre de **minutes** qui l'en séparent *(votre Dijkstra)*.

Écrivez `mesures(station, nb_reperes)` de façon à pouvoir ajouter des repères
ensuite, et construisez `X_1repere` ainsi que `y`.

In [21]:
REPERES = ["Berri-UQAM", "Snowdon", "Honore-Beaugrand"]

stations_retenues = [s for s in sorted(reseau) if infos_station[s]["ligne"] != "jaune"]
y = [infos_station[s]["ligne"] for s in stations_retenues]
print(len(stations_retenues), "stations retenues")

# Q6.1
def mesures(station, nb_reperes):
    resultat = []
    
    nb_voisins = len(voisins(reseau, station))
    resultat.append(nb_voisins)
    
    durees = [minutes for voisin, minutes in reseau[station]]
    moyenne = sum(durees) / len(durees)
    resultat.append(moyenne)
    
    plus_long = max(durees)
    resultat.append(plus_long)
    
    for repere in REPERES[:nb_reperes]:
        distance, precedent = bfs(reseau, repere)
        resultat.append(distance[station])
        
    distance, precedent = dijkstra(reseau, REPERES[0])
    resultat.append(distance[station])
    
    return resultat

X_1repere =  [mesures(station, 1) for station in stations_retenues]

print(len(X_1repere))
print(X_1repere[0])





66 stations retenues
66
[2, 3.0, 3, 9, 17]


> **Observation — obligatoire.** Lesquelles de vos cinq mesures viennent des fichiers fournis, et lesquelles viennent de votre propre code ? Qui a choisi les repères ?
>
> nb de voisins: il y avait déjà une fonction qui faisait ca: voisins(reseau, station)
>   reseau a été construit à partir des données du fichier fourni: donnees/metro_montreal.csv
>
> pour la moyenne des tronçons, la durée du plus long tronçon : les données viennent des fichiers, mais les calculs sont fait par mon code. 
>
> ont été utilisées les fonctions bfs(reseau, depart) et dijkstra(reseau, depart) qui utilisent réseau, lui-même déduit des données > des fichiers. ils calculent de nouvelles mesures à partir du réseau.

>
> Les repères étaient fournis dans une liste dans le starter code, fourni par le professeur.

**Q6.2** — Comptez combien de stations portent chaque étiquette, et déduisez-en la
note qu'obtiendrait un modèle qui répondrait **toujours la même chose**.

In [22]:
# Q6.2
nb_verte = y.count("verte")
nb_bleue = y.count("bleue")
nb_orange = y.count("orange")
nb_stations = len(stations_retenues)

pc_verte = (nb_verte * 100) / nb_stations
pc_bleue = (nb_bleue * 100) / nb_stations
pc_orange = (nb_orange * 100) / nb_stations 

print(f"{'Ligne':<10}{'Nombre de stations':<20}{'Pourcentage':<12}%")
print('-' * 42)
print(f"{'Verte':<10}{nb_verte:<20}{pc_verte:<12.2f}%")
print(f"{'Bleue':<10}{nb_bleue:<20}{pc_bleue:<12.2f}%")
print(f"{'Orange':<10}{nb_orange:<20}{pc_orange:<12.2f}%")

Ligne     Nombre de stations  Pourcentage %
------------------------------------------
Verte     27                  40.91       %
Bleue     10                  15.15       %
Orange    29                  43.94       %


> **Observation — obligatoire.** Notez ce pourcentage. C'est la barre que votre modèle devra dépasser pour servir à quelque chose.
>
>En calculant les pourcentages, on obtient la réponse à la question à savoir quelle note obtiendrait un modèle qui dirait toujours la même chose.
>
> par exemple, pour "Orange": 43.94% des stations du réseau sont des stations sur la ligne orange.  
> un modèle qui répondrait toujours la même chose aurait raison dans 43.94% des cas.
> 
> 43.94% est donc la barre à dépasser pour que le modèle soit meilleur.

## Partie 7 — Entraîner un modèle, et le noter honnêtement

**Q7.1** — Coupez en deux paquets (70 / 30), entraînez un `DecisionTreeClassifier`,
et affichez **deux** notes : sur les stations apprises, et sur les stations jamais
vues.

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Q7.1

X_train, X_test, y_train, y_test = train_test_split(
    X_1repere, 
    y, 
    test_size=0.3, 
    random_state=0
    )

arbre = DecisionTreeClassifier(max_depth=2, random_state=0) 
arbre.fit(X_train, y_train)

print("entrainement :", round(arbre.score(X_train, y_train) * 100, 1), "%")
print("test :", round(arbre.score(X_test, y_test) * 100, 1), "%")



entrainement : 58.7 %
test : 30.0 %


> **Observation — obligatoire.** Vos deux notes sont très différentes. Comment s'appelle ce phénomène, et laquelle des deux est honnête ?
>
> *(remplacez ce texte par vos phrases)*

**Q7.2** — Refaites la même chose avec **2** puis **3** repères. Affichez les trois
notes côte à côte.

In [24]:
# Q7.2

X_2reperes = [mesures(station, 2) for station in stations_retenues]

X_train, X_test, y_train, y_test = train_test_split(
    X_2reperes, 
    y, 
    test_size=0.3, 
    random_state=0
    )

arbre = DecisionTreeClassifier(max_depth=2, random_state=0) 
arbre.fit(X_train, y_train)

print("entrainement :", round(arbre.score(X_train, y_train) * 100, 1), "%")
print("test :", round(arbre.score(X_test, y_test) * 100, 1), "%")

# entrainement : 60.9 %
# test : 55.0 %



# =========== 3 REPERES =======================

X_3reperes = [mesures(station, 3) for station in stations_retenues]

X_train, X_test, y_train, y_test = train_test_split(
    X_3reperes, 
    y, 
    test_size=0.3, 
    random_state=0
    )

arbre = DecisionTreeClassifier(max_depth=2, random_state=0) 
arbre.fit(X_train, y_train)

print("entrainement :", round(arbre.score(X_train, y_train) * 100, 1), "%")
print("test :", round(arbre.score(X_test, y_test) * 100, 1), "%")

# entrainement : 65.2 %
# test : 60.0 %


entrainement : 60.9 %
test : 55.0 %
entrainement : 65.2 %
test : 60.0 %


> **Observation — obligatoire.** Qu'est-ce qui change, et pourquoi un seul repère ne suffisait-il pas ? Pensez à deux stations situées de part et d'autre du repère.
>
> avec DEUX repères, le résultat des stations jamais vues passe de 30% à 55%
> 55% dépasse la baseline de 43.94%
> ca s'explique par le fait que deux distances donnent plus d'info sur la position de la station dans le réseau.
>
> avec TROIS repères. 
> entrainement : 65.2 %
> test : 60.0 %
> => avec davantage de repères, la note sur les stations jamais vues augmente


| Repères | Entraînement | Test |
|---|---:|---:|
| 1 repère | 58,7 % | 30,0 % |
| 2 repères | 60,9 % | 55,0 % |
| 3 repères | 65,2 % | 60,0 % |

**Q7.3** — Sur vos meilleures mesures, comparez un **arbre seul** à une **forêt
aléatoire**. Relancez chacun avec `random_state` valant 0, 1, 2, 3 puis 4, et
affichez pour chaque modèle ses cinq notes, leur **moyenne** et l'**écart** entre la
plus haute et la plus basse.

In [31]:
# Q7.3
X_3reperes = [mesures(station, 3) for station in stations_retenues]

random_states = [0, 1, 2, 3, 4]

scores_arbre = []

for random_state in random_states:
    X_train, X_test, y_train, y_test = train_test_split(
        X_3reperes, y, test_size=0.3, random_state=0
    )

    arbre = DecisionTreeClassifier(max_depth=2, random_state=random_state)
    arbre.fit(X_train, y_train)

    score = arbre.score(X_test, y_test)
    scores_arbre.append(score)

print(scores_arbre)


# ================== SCORES FORET ===========================

scores_foret = []

for random_state in random_states:
    foret = RandomForestClassifier(random_state=random_state)
    foret.fit(X_train, y_train)

    score = foret.score(X_test, y_test)
    scores_foret.append(score)

print(scores_foret)




# =============== MOYENNE ET ECART =============================

moyenne_arbre = sum(scores_arbre) / len(scores_arbre)
ecart_arbre = max(scores_arbre) - min(scores_arbre)

moyenne_foret = sum(scores_foret) / len(scores_foret)
ecart_foret = max(scores_foret) - min(scores_foret)

print("Arbre")
print("Notes :", scores_arbre)
print("Moyenne :", moyenne_arbre)
print("Écart :", ecart_arbre)

print()
print("Forêt")
print("Notes :", scores_foret)
print(f"Moyenne : {moyenne_foret:.4f}")
print(f"Écart : {ecart_foret:.4f}")


[0.6, 0.6, 0.6, 0.6, 0.6]
[0.65, 0.75, 0.75, 0.6, 0.65]
Arbre
Notes : [0.6, 0.6, 0.6, 0.6, 0.6]
Moyenne : 0.6
Écart : 0.0

Forêt
Notes : [0.65, 0.75, 0.75, 0.6, 0.65]
Moyenne : 0.6800
Écart : 0.1500


> **Observation — obligatoire.** La forêt vous a-t-elle aidés ? Appuyez-vous sur la moyenne ET sur l'écart, puis proposez une explication liée au nombre d'exemples dont vous disposez.
>
> 
l'arbre a une moyenne de 60% et un écart de 0.

La forêt a une moyenne de 68% et un écart de 15%

| Modèle | Notes | Moyenne | Écart |
|---|---|---:|---:|
| Arbre | 60 %, 60 %, 60 %, 60 %, 60 % | 60 % | 0 % |
| Forêt aléatoire | 65 %, 75 %, 75 %, 60 %, 65 % | 68 % | 15 % |

oui, on peut dire que la forêt a aidé: sa moyenne est de 8% de plus que pour l'arbre.

l'écart de 15% peut être expliqué par la variation du random state.

à noter que le pool de données est relativement petit: 66 stations en tout, seulement 46 stations comme données d'entraînement...



---
## Partie 8 — Vos conclusions

Répondez en Markdown, **en phrases complètes, en français**. Comptez deux ou trois
phrases par question, et **citez vos propres chiffres**.

**Sur le réseau**

1. Le réseau est-il un **arbre** ? Justifiez avec le nombre de stations et de liens.
2. Quelle station est la mieux connectée, et **pourquoi** celle-là ?
3. Sur vos trois paires, le BFS et Dijkstra donnent-ils souvent la même réponse ?
   Reliez votre réponse au décompte des durées.
4. Décrivez **une** paire où ils diffèrent, et expliquez ce que chacun a optimisé.

**Sur le modèle**

5. Avec un seul repère, quelle note obtenez-vous, et comment se compare-t-elle au
   modèle qui répond toujours la même chose ? Qu'est-ce que ça vous apprend ?
6. Qu'est-ce qui change avec 2 puis 3 repères, et **pourquoi** ?
7. Pourquoi la note sur les stations jamais vues est-elle la seule honnête ?
8. La forêt aléatoire vous a-t-elle aidés ? Appuyez-vous sur les cinq notes et sur
   l'écart.

**Sur l'ensemble**

9. Vos données de la partie 6 viennent de vos propres fonctions. Quel risque cela
   pose-t-il si votre BFS ou votre Dijkstra contient une erreur ?
10. Nommez **une** limite de nos données par rapport au vrai métro, et **une**
    mesure supplémentaire que vous auriez aimé avoir pour le modèle.

### Vos réponses

*(remplacez ce texte)*

1.  si on applique la règle d'un arbre, il doit avoir un lien de moins que le nombre de sommet. 
   Or, ici on a 68 sommets et 69 liens
   donc ce n'est pas un arbre

2.  La station Berri-UQAM est la mieux connectée, tel que montré à la question Q2.2 avec la fonction def nbVoisins(reseau):
   la fonction a trouvé que cette station était celle qui avait le plus de voisins.  Ceci s'explique par le fait que 3 lignes s'intersectent à ce noeud.

3. On avait trouvé que les deux algorithmes trouvaient les mêmes réponses dans 2/3 des cas.  Dans un cas, celui de Parc - Angrignon, Dijkstra avait trouvé le chemin de durée minimale, tandis que BFS a retourné le nombre minimal de stations.  Ceci s'explique par le fait que les chemins sont composés de tronçons de durée différentes. 

4. Dans le cas où ils diffère, soit le cas de Parc -- Angrignon, 
      le BFS a retourné 17 stations en 41 minutes
      Dijkstra a retourné 22 stations en 37 minutes.
   ces résultats montrent clairement ce que chaque algorithme privilégie: 
   BFS: le nombre minimal de stations, 
   Dijkstra: la durée minimale de la somme des tronçons.

5.  On a trouvé qu'avec le modèle avec 1 repère on obtenait 30% au test. Comparé au modèle qui obtendrait 43.94% s'il répondait toujours 'orange', ca ne représente pas une amélioration, c'est même bien en dessous. Les résultats bas de 30% montrent qu'il généralisait mal aux stations du test, ce qui est signe de suraprentissage entre l'entrainement (58.7%) et le test(30%). 

6. la performance du modèle augmente avec 2 et 3 repères, ce qui montre que le nombre de repères a un impact.  
   un seul repère ne donnait que 30%, 
   deux repères: 55%
   trois repères: 60%
   un seul repère ne donne qu'une seule référence de position, et augmenter le nombre de repères donne davantage d'information sur la position des stations. 

7. Les données d'entraînement servent à apprendre au modèle, tandis que les données de test sont constituées de stations que le modèle n'a jamais vues. La note sur les stations jamais vues est la seule honnête parce qu'elle montre comment le modèle se comporte sur de nouvelles données, après son entraînement.

8.  l'arbre avait une moyenne de 60% et un écart de 0%.  notes: 60 %, 60 %, 60 %, 60 %, 60 %
    la forêt avait une moyenne de 68% et un écart de 15%. notes: 65 %, 75 %, 75 %, 60 %, 65 %
      la forêt a aidé car la moyenne était supérieure.
      la forêt est moins stable, parce que son écart était de 15% comparé à celui de l,arbre, qui est de 0%.

9. on utilise bfs pour calculer les distances, et dijkstra pour calculer les durées. s'il y a une erreur dans ces fonctions, ces données seront fournies au modèle et celui-ci apprendra à partir de données erronées. 

10. C'est une version simplifiée du vrai métro de Montréal (heureusement !). En effet, il ne tient pas compte du temps de marche, du temps d'attente du métro ni du temps nécessaire pour changer aux correspondances.

Comme mesure supplémentaire, je crois que ce ne serait pas coûteux de rendre les durées des tronçons plus précises. Au lieu de 1, 2 ou 3 minutes, on pourrait ajouter les secondes, par exemple 2 minutes et 43 secondes. Les fonctions que nous avons faites pourraient aisément s'en accommoder.
Aussi, il y a le problème que quand une station a déjà été découverte par un autre chemin, elle n'est pas répétée; les cycles sont éliminés. ca peut donner une représentation irréaliste du réseau quand il y a des correspondances entre deux lignes, comme on l'a vu à la question Q3.3, en utilisant graphviz. le graphe donne le parcours de l,algorithme, mais pas le vrai plan du métro.


---
## Déclaration d'utilisation de l'intelligence artificielle

**Obligatoire, même si vous n'en avez pas utilisé.** Une déclaration absente ou
fausse vaut **0 pour l'ensemble du TP**.

L'IA est permise **pour apprendre**, pas pour produire votre travail. Relisez les
règles de l'énoncé avant de remplir cette section.

**Avez-vous utilisé un outil d'IA ?** _oui 

**Si oui, remplissez le tableau :**

| Outil | Pour quelle question | Ce que vous lui avez demandé | Lien de partage de la conversation |
|---|---|---|---|
|  |Q2  |explication code  |https://chatgpt.com/share/6ac70039-8e00-83e9-9ee4-7243bbd8b6fc  | 
|  | Q3 | explication code | https://chatgpt.com/share/6ac7e054-caa8-83e9-93a7-6cf07c487e51 |

**Confirmez, en écrivant « oui » :**

- Je n'ai **pas** collé l'énoncé du TP, en tout ou en partie, dans une conversation
  avec une IA. → oui, je ne l'ai pas fait; 
- Je n'ai **pas** collé le contenu des fichiers de données. → oui, je ne l'ai pas fait
- Je comprends chaque ligne de code que je remets et je peux l'expliquer à l'oral.
  → _____

---
**Qui a fait quoi :** *(obligatoire si vous êtes deux)*

---
## Avant de remettre

**Vingt points du TP ne portent pas sur vos réponses** : dix sur la tenue de ce
carnet, dix sur votre usage de Git. Relisez les sections 3 et 4 de l'énoncé avant de
déposer votre travail.

**Le carnet.** Faites **Kernel → Redémarrer et tout exécuter** en dernier, et
vérifiez qu'aucune cellule ne produit d'erreur. Puis relisez-le comme si vous ne
l'aviez pas écrit : est-ce qu'on s'y retrouve ? Est-ce que chaque nombre affiché
s'explique sans aller lire le code ? Reste-t-il des cellules d'essai, des variables
au nom obscur, des observations expédiées ? Le français est-il relu ?

**Le dépôt.** Le nom de votre branche et vos messages de commit suivent la
convention vue en cours. Votre historique doit montrer un travail construit au fil du
temps — et, si vous êtes deux, la contribution des deux.

**L'identification.** Nom, coéquipier et groupe en première cellule, « qui a fait
quoi » à la fin, déclaration d'IA remplie même si la réponse est « non », et le
dossier `donnees/` à côté du carnet.

---

*Ensuite : `Publish branch`, `Create Pull Request`, fusion dans `main`, et le lien du
dépôt sur Omnivox.*